# Pre subida al Postgres

### Archivos de entrada
| Archivo | Contenido |
|---|---|
| `esea_meta_demos.part1.csv` | Metadata de partidas y rondas |
| `esea_master_dmg_demos.part1.csv` | Eventos de daño por tick |
| `esea_master_kills_demos.part1.csv` | Eventos de kill por tick |

### Tablas
**Entidades:** `Partida`, `Ronda`, `Arma`, `Jugador`, `Ticks`  
**Relaciones:** `Participacion`, `Danno` (danno en vez de daño pq se puede chingar el postgres)

In [12]:
import pandas as pd
dmeta  = pd.read_csv("esea_meta_demos.part1.csv")
ddmg   = pd.read_csv("esea_master_dmg_demos.part1.csv")
dkills = pd.read_csv("esea_master_kills_demos.part1.csv")

In [14]:
dmeta.head()

,file,map,round,start_seconds,end_seconds,winner_team,winner_side,round_type,ct_eq_val,t_eq_val
0,esea_match_13770997.dem,de_overpass,1,94.30782,160.9591,Hentai Hooligans,Terrorist,PISTOL_ROUND,4300,4250
1,esea_match_13770997.dem,de_overpass,2,160.95910,279.3998,Hentai Hooligans,Terrorist,ECO,6300,19400
2,esea_match_13770997.dem,de_overpass,3,279.39980,341.0084,Hentai Hooligans,Terrorist,SEMI_ECO,7650,19250
3,esea_match_13770997.dem,de_overpass,4,341.00840,435.4259,Hentai Hooligans,Terrorist,NORMAL,24900,23400
4,esea_match_13770997.dem,de_overpass,5,435.42590,484.2398,Animal Style,CounterTerrorist,ECO,5400,20550


---
## Entidades
### 1. Partida

In [10]:
Partida = (
    dmeta[["file", "map"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

Partida.head()

,file,map
0,esea_match_13770997.dem,de_overpass
1,esea_match_13779704.dem,de_cache
2,esea_match_13779769.dem,de_inferno
3,esea_match_13779770.dem,de_mirage
4,esea_match_13779771.dem,de_inferno


### 2. Ronda
Una fila por ronda. `esea_meta_demos` ya tiene una fila por ronda, solo se renombran columnas.

Se incluyen `ct_eq_val` y `t_eq_val` (economía de cada equipo al inicio de la ronda) porque son parte del análisis principal del proyecto.

In [ ]:
Ronda = (
    dmeta[["file", "round", "start_seconds", "end_seconds",
           "winner_team", "winner_side", "round_type",
           "ct_eq_val", "t_eq_val"]]
    .copy()
    .rename(columns={
        "file":  "archivo",
        "round": "numero"
    })
)

Ronda.head()

,archivo,numero,start_seconds,end_seconds,winner_team,winner_side,round_type,ct_eq_val,t_eq_val
0,esea_match_13770997.dem,1,94.30782,160.9591,Hentai Hooligans,Terrorist,PISTOL_ROUND,4300,4250
1,esea_match_13770997.dem,2,160.95910,279.3998,Hentai Hooligans,Terrorist,ECO,6300,19400
2,esea_match_13770997.dem,3,279.39980,341.0084,Hentai Hooligans,Terrorist,SEMI_ECO,7650,19250
3,esea_match_13770997.dem,4,341.00840,435.4259,Hentai Hooligans,Terrorist,NORMAL,24900,23400
4,esea_match_13770997.dem,5,435.42590,484.2398,Animal Style,CounterTerrorist,ECO,5400,20550


In [17]:
Ronda.columns

Index(['archivo', 'numero', 'start_seconds', 'end_seconds', 'winner_team',
       'winner_side', 'round_type', 'ct_eq_val', 't_eq_val'],
      dtype='object')

### 3. Arma


In [4]:
Arma = (
    ddmg[["wp", "wp_type"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
    .rename(columns={
        "wp":      "nombre",
        "wp_type": "tipo"
    })
)

Arma.head()

,nombre,tipo
0,Unknown,Unkown
1,USP,Pistol
2,Glock,Pistol
3,HE,Grenade
4,Deagle,Pistol


### 4. Jugador
Los jugadores aparecen como `att_id` y `vic_id` en cada fila de daño.

Se unen ambas columnas, se eliminan duplicados y se filtra `id = 0` que representa al daño por caída, fuego, etc.

In [5]:
Jugador = (
    pd.concat([
        ddmg[["att_id"]].rename(columns={"att_id": "id"}),
        ddmg[["vic_id"]].rename(columns={"vic_id": "id"})
    ], ignore_index=True)
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

Jugador = Jugador[Jugador["id"] != 0].copy()
Jugador["id"] = Jugador["id"].astype("int64")

Jugador.head()

,id
1,76561198048742997
2,76561198055054795
3,76561198082200410
4,76561198055191021
5,76561198037331400


### 5. Ticks
Se reconstruye cruzando:
- `ddmg` → estado de la bomba (`is_bomb_planted`, `bomb_site`) por tick
- `dkills` → jugadores vivos (`ct_alive`, `t_alive`) por tick

 `outer join` para no perder ticks que aparezcan solo en una de las dos fuentes.

In [6]:
# Estado de bomba: un tick puede tener múltiples eventos de daño
# pero el estado de la bomba es el mismo en todos → drop_duplicates antes del merge
ticks_bomba = (
    ddmg[["file", "round", "tick", "seconds", "is_bomb_planted", "bomb_site"]]
    .drop_duplicates(subset=["file", "round", "tick"])
)

# Jugadores vivos: solo cambia en ticks donde hubo un kill
ticks_vivos = (
    dkills[["file", "round", "tick", "ct_alive", "t_alive"]]
    .drop_duplicates(subset=["file", "round", "tick"])
)

Ticks = (
    pd.merge(ticks_bomba, ticks_vivos, on=["file", "round", "tick"], how="outer")
    .rename(columns={
        "file":  "archivo",
        "round": "numero_ronda",
        "tick":  "numero",
        "seconds": "tiempo"
    })
)

Ticks.head()

,archivo,numero_ronda,numero,tiempo,is_bomb_planted,bomb_site,ct_alive,t_alive
0,esea_match_13770997.dem,1,14372,111.8476,False,NaN,NaN,NaN
1,esea_match_13770997.dem,1,15972,124.3761,False,NaN,NaN,NaN
2,esea_match_13770997.dem,1,16058,125.0495,False,NaN,5.0,4.0
3,esea_match_13770997.dem,1,16066,125.1121,False,NaN,NaN,NaN
4,esea_match_13770997.dem,1,16108,125.4410,False,NaN,NaN,NaN


In [18]:
Ticks.columns

Index(['archivo', 'numero_ronda', 'numero', 'tiempo', 'is_bomb_planted',
       'bomb_site', 'ct_alive', 't_alive'],
      dtype='object')

---
## Relaciones
### 6. Participacion
Qué jugador participó en qué partida y para qué equipo.

Un jugador cambia de lado exactamente en la ronda 16 (CT ↔ T), por lo que aparece en ambos lados dentro de la misma partida. Se usa `mode()` para quedarse con el equipo más frecuente, que corresponde al equipo principal del jugador en esa partida.

In [7]:
Participacion = (
    ddmg[["file", "att_id", "att_team"]]
    [ddmg["att_id"] != 0]
    .groupby(["file", "att_id"])["att_team"]
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={
        "file":     "archivo",
        "att_id":   "id_jugador",
        "att_team": "equipo"
    })
)

Participacion.head()

,archivo,id_jugador,equipo
0,esea_match_13770997.dem,76561197961009213,Hentai Hooligans
1,esea_match_13770997.dem,76561198037331400,Hentai Hooligans
2,esea_match_13770997.dem,76561198048742997,Animal Style
3,esea_match_13770997.dem,76561198055054795,Animal Style
4,esea_match_13770997.dem,76561198055191021,Animal Style


In [19]:
Participacion.columns

Index(['archivo', 'id_jugador', 'equipo'], dtype='object')

### 7. Danno
Cada evento de daño entre dos jugadores en un tick específico.

La columna `mata` sale cruzando con `dkills`: si existe un kill en el mismo `(file, round, tick)`, el evento de daño resultó en muerte.

In [8]:
kills_key = (
    dkills[["file", "round", "tick"]]
    .drop_duplicates()
    .assign(mata=True)
)

Danno = (
    ddmg[[
        "file", "round", "tick",
        "att_id", "vic_id",
        "att_side", "vic_side",    
        "wp", "hitbox",
        "hp_dmg", "arm_dmg",
        "att_pos_x", "att_pos_y",
        "vic_pos_x", "vic_pos_y"
    ]]
    .copy()
    [ddmg["att_id"] != 0]
)

Danno = (
    pd.merge(Danno, kills_key, on=["file", "round", "tick"], how="left")
)
Danno["mata"] = Danno["mata"].infer_objects(copy=False).fillna(False)

Danno = Danno.rename(columns={
    "file":    "partida",
    "round":   "ronda",
    "att_id":  "att",
    "vic_id":  "vic",
    "wp":      "arma",
    "hp_dmg":  "danno_hp",
    "arm_dmg": "danno_armor",
    "att_side": "lado_att",
    "vic_side": "lado_vic"
})

Danno.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_28232\1719013826.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Danno["mata"] = Danno["mata"].infer_objects(copy=False).fillna(False)


,partida,ronda,tick,att,vic,lado_att,lado_vic,arma,hitbox,danno_hp,danno_armor,att_pos_x,att_pos_y,vic_pos_x,vic_pos_y,mata
0,esea_match_13770997.dem,1,15972,76561198048742997,76561198082200410,CounterTerrorist,Terrorist,USP,Stomach,18,9,-1499.6900,63.33829,-669.5558,-79.769570,False
1,esea_match_13770997.dem,1,16058,76561198055054795,76561197961009213,CounterTerrorist,Terrorist,USP,Head,100,0,-1066.8740,3.44563,-614.1868,-91.707770,True
2,esea_match_13770997.dem,1,16066,76561198082200410,76561198055054795,Terrorist,CounterTerrorist,Glock,RightArm,12,7,-747.3146,-49.32681,-1065.5560,9.381622,False
3,esea_match_13770997.dem,1,16108,76561198048742997,76561198082200410,CounterTerrorist,Terrorist,USP,Chest,15,7,-1501.8610,49.19798,-748.4188,-53.469220,False
4,esea_match_13770997.dem,1,16188,76561198082200410,76561198048742997,Terrorist,CounterTerrorist,Glock,Head,94,0,-756.2186,-80.94859,-1500.0780,60.658150,False


In [20]:
Danno.columns

Index(['partida', 'ronda', 'tick', 'att', 'vic', 'lado_att', 'lado_vic',
       'arma', 'hitbox', 'danno_hp', 'danno_armor', 'att_pos_x', 'att_pos_y',
       'vic_pos_x', 'vic_pos_y', 'mata'],
      dtype='object')

---
## Exportar CSVs

In [9]:
import os
os.makedirs("output", exist_ok=True)

tablas = {
    "partida":       Partida,
    "ronda":         Ronda,
    "arma":          Arma,
    "jugador":       Jugador,
    "ticks":         Ticks,
    "participacion": Participacion,
    "danno":         Danno
}

for nombre, df in tablas.items():
    ruta = f"output/{nombre}.csv"
    df.to_csv(ruta, index=False)
    print(f"{nombre:15s} → {ruta}  ({len(df):,} filas)")

partida         → output/partida.csv  (8,527 filas)
ronda           → output/ronda.csv  (215,919 filas)
arma            → output/arma.csv  (42 filas)
jugador         → output/jugador.csv  (17,878 filas)
ticks           → output/ticks.csv  (5,778,260 filas)
participacion   → output/participacion.csv  (86,869 filas)
danno           → output/danno.csv  (5,933,077 filas)


-- =====================================================
-- TIPOS PERSONALIZADOS
-- =====================================================

CREATE TYPE COORDENADA AS (
    x FLOAT,
    y FLOAT
);

CREATE TYPE LADO AS ENUM ('CounterTerrorist', 'Terrorist');

CREATE TYPE SITIO_BOMBA AS ENUM ('A', 'B');

CREATE TYPE HITBOX AS ENUM (
    'Head', 'Chest', 'Stomach', 'RightArm', 'LeftArm',
    'RightLeg', 'LeftLeg', 'Generic', 'Neck'
);

CREATE TYPE TIPO_RONDA AS ENUM (
    'PISTOL_ROUND', 'ECO', 'SEMI_ECO', 'NORMAL', 'FORCE_BUY'
);

-- =====================================================
-- ENTIDADES
-- =====================================================

CREATE TABLE Partida (
    archivo VARCHAR(40) PRIMARY KEY,
    mapa    VARCHAR(30) CHECK (mapa IN (
        'de_dust2', 'de_mirage', 'de_inferno', 'de_nuke',
        'de_overpass', 'de_cache', 'de_train', 'de_cbble'
    ))
);

CREATE TABLE Ronda (
    archivo       VARCHAR(40) REFERENCES Partida(archivo) ON DELETE CASCADE,
    numero        INT,
    start_seconds FLOAT,
    end_seconds   FLOAT,
    winner_team   VARCHAR(30),
    winner_side   LADO,
    round_type    TIPO_RONDA,
    ct_eq_val     INT,
    t_eq_val      INT,
    PRIMARY KEY (archivo, numero)
);

CREATE TABLE Ticks (
    archivo         VARCHAR(40),
    numero_ronda    INT,
    numero          INT,
    ct_alive        INT,
    t_alive         INT,
    is_bomb_planted BOOL,
    bomb_site       SITIO_BOMBA,
    tiempo          FLOAT,
    FOREIGN KEY (archivo, numero_ronda) REFERENCES Ronda(archivo, numero) ON DELETE CASCADE,
    PRIMARY KEY (archivo, numero_ronda, numero)
);

CREATE TABLE Jugador (
    id BIGINT PRIMARY KEY
);

CREATE TABLE Arma (
    nombre VARCHAR(30) PRIMARY KEY,
    tipo   VARCHAR(20)
);

-- =====================================================
-- RELACIONES
-- =====================================================

CREATE TABLE Participacion (
    archivo    VARCHAR(40) REFERENCES Partida(archivo) ON DELETE CASCADE,
    id_jugador BIGINT REFERENCES Jugador(id) ON DELETE CASCADE,
    equipo     VARCHAR(30),
    PRIMARY KEY (archivo, id_jugador)
);

CREATE TABLE Danno (
    id           SERIAL PRIMARY KEY,
    att          BIGINT REFERENCES Jugador(id) ON DELETE CASCADE,
    vic          BIGINT REFERENCES Jugador(id) ON DELETE CASCADE,
    arma         VARCHAR(30) REFERENCES Arma(nombre) ON DELETE CASCADE,
    partida      VARCHAR(40),
    ronda        INT,
    tick         INT,
    lado_att     LADO,
    lado_vic     LADO,
    hitbox       HITBOX,
    danno_hp     INT,
    danno_armor  INT,
    att_pos_x    FLOAT,
    att_pos_y    FLOAT,
    vic_pos_x    FLOAT,
    vic_pos_y    FLOAT,
    mata         BOOL,
    FOREIGN KEY (partida, ronda, tick) REFERENCES Ticks(archivo, numero_ronda, numero) ON DELETE CASCADE
);